# One-Step RMSE as a Function of Altitude

Mean-square error of $U$ after one forecast step, as a function of altitude $z$, averaged over many initial conditions — computed separately for State A and State B. Compares the HM stochastic model vs the VAE emulator.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
DATA_PATH      = '/home/fabio/work/HM_and_AI_models/VAE_Model/data/long_run_310k.npy'
EMU_MODEL_PATH = '/home/fabio/work/HM_and_AI_models/VAE_Model/data/best_weights/checkpoint_11'
SAVE_DIR       = os.path.dirname(os.path.abspath('__file__'))

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

LENGTH_SCALE   = 2.5e5
TIME_SCALE     = 24 * 3600.0
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE

NZ = 26
VERTICAL_LEVELS = np.linspace(0, 70e3, NZ + 1)[1:-1] / 1000

UPPER_BOUND = 53.8 / VELOCITY_SCALE
LOWER_BOUND = 21.44 / VELOCITY_SCALE

LATENT_DIM    = 32
OUTPUT_DIM    = 75
CONDITION_DIM = 50
NUM_NEURONS   = 1024

NUM_ICS_1STEP  = 200
NUM_ENS_1STEP  = 500

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(OUTPUT_DIM, NUM_NEURONS)
        self.fc2 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc3 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc4 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc5 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc6 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc_mu     = nn.Linear(NUM_NEURONS, LATENT_DIM)
        self.fc_logvar = nn.Linear(NUM_NEURONS, LATENT_DIM)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x)) + x
        x = torch.relu(self.fc3(x)) + x
        x = torch.relu(self.fc4(x)) + x
        x = torch.relu(self.fc5(x)) + x
        x = torch.relu(self.fc6(x)) + x
        return self.fc_mu(x), self.fc_logvar(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(LATENT_DIM + CONDITION_DIM, NUM_NEURONS)
        self.fc2 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc3 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc4 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc5 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc6 = nn.Linear(NUM_NEURONS, NUM_NEURONS)
        self.fc_output = nn.Linear(NUM_NEURONS, OUTPUT_DIM)

    def forward(self, z, cond):
        x = torch.cat((z, cond), dim=1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x)) + x
        x = torch.relu(self.fc3(x)) + x
        x = torch.relu(self.fc4(x)) + x
        x = torch.relu(self.fc5(x)) + x
        x = torch.relu(self.fc6(x)) + x
        return self.fc_output(x)

class ConditionalVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def decode(self, z, cond):
        return self.decoder(z, cond)

    def forward(self, x, cond):
        mu, logvar = self.encoder(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decode(z, cond), mu, logvar

In [ ]:
print("Loading data...")
real_data = np.load(DATA_PATH)
psi       = real_data[:, 1, :]
N         = len(psi)

mean_psi = np.mean(psi, axis=0)
std_psi  = np.std(psi, axis=0)

zonal_wind_ref = psi[:, 63]

U_all    = psi[:, 50:75] * VELOCITY_SCALE
U_clim_A = U_all[zonal_wind_ref > UPPER_BOUND].mean(axis=0)
U_clim_B = U_all[zonal_wind_ref < LOWER_BOUND].mean(axis=0)

valid = np.arange(N - 1)

In [ ]:
from holton_mass import HoltonMassModel
print("Initialising HM model...")
hm_model = HoltonMassModel(HoltonMassModel.default_config())
t_save   = np.array([0.0, 1.0])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vae    = ConditionalVAE().to(device)
vae.load_state_dict(torch.load(EMU_MODEL_PATH, map_location=device, weights_only=True))
vae.eval()
print(f"Emulator loaded  (device: {device})")

In [ ]:
idx_A = valid[zonal_wind_ref[valid] > UPPER_BOUND]
idx_B = valid[zonal_wind_ref[valid] < LOWER_BOUND]

chosen_A = np.random.choice(idx_A, NUM_ICS_1STEP, replace=False)
chosen_B = np.random.choice(idx_B, NUM_ICS_1STEP, replace=False)

def one_step_mse_profile(chosen_idx, label):
    """Return (25,) MSE profile [m/s]^2 for HM and emulator over chosen ICs."""
    hm_se_accum  = np.zeros(25)
    emu_se_accum = np.zeros(25)

    for k, idx in enumerate(chosen_idx):
        if k % 50 == 0:
            print(f"  State {label}: IC {k}/{NUM_ICS_1STEP}")

        state   = psi[idx]
        truth   = psi[idx + 1]
        U_truth = truth[50:75] * VELOCITY_SCALE

        # --- HM ensemble ---
        x0    = np.tile(state, (NUM_ENS_1STEP, 1))
        x_ens = hm_model.integrate_euler_maruyama(x0, t_save, seed=None)
        U_hm_mean = x_ens[1, :, 50:75].mean(axis=0) * VELOCITY_SCALE
        hm_se_accum += (U_hm_mean - U_truth) ** 2

        # --- Emulator ensemble ---
        x_norm = ((state - mean_psi) / std_psi)[:CONDITION_DIM].astype(np.float32)
        cond   = torch.tensor(x_norm).unsqueeze(0).expand(NUM_ENS_1STEP, -1).to(device)
        z      = torch.randn(NUM_ENS_1STEP, LATENT_DIM, device=device)
        with torch.no_grad():
            y = vae.decode(z, cond).cpu().numpy()
        U_emu_mean = (y * std_psi + mean_psi)[:, 50:75].mean(axis=0) * VELOCITY_SCALE
        emu_se_accum += (U_emu_mean - U_truth) ** 2

    return hm_se_accum / NUM_ICS_1STEP, emu_se_accum / NUM_ICS_1STEP

mse_hm_A, mse_emu_A = one_step_mse_profile(chosen_A, 'A')
mse_hm_B, mse_emu_B = one_step_mse_profile(chosen_B, 'B')

rmse_hm_A,  rmse_emu_A  = np.sqrt(mse_hm_A),  np.sqrt(mse_emu_A)
rmse_hm_B,  rmse_emu_B  = np.sqrt(mse_hm_B),  np.sqrt(mse_emu_B)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharey=True)

for i, (ax, rmse_hm, rmse_emu, title, U_bg) in enumerate([
    (axes[0], rmse_hm_A, rmse_emu_A, 'State A', U_clim_A),
    (axes[1], rmse_hm_B, rmse_emu_B, 'State B', U_clim_B),
]):
    ax.plot(rmse_hm,  VERTICAL_LEVELS, 'o-', color='tab:green', linewidth=2,
            markersize=5, label='HM Model')
    ax.plot(rmse_emu, VERTICAL_LEVELS, 's-', color='tab:red',   linewidth=2,
            markersize=5, label='Emulator')
    ax.set_xlabel(r'One-step RMSE of $U$  [m/s]', fontsize=13)
    ax.set_title(title, fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.tick_params(labelsize=11)

    if i == 0:
        ax.legend(fontsize=11, loc='upper left')

    ax2 = ax.twiny()
    ax2.plot(U_bg, VERTICAL_LEVELS, 'k--', linewidth=1.8,
             alpha=0.6, label=r'$\overline{U}$')
    ax2.tick_params(axis='x', labelsize=10)
    if i == 0:
        ax2.legend(fontsize=10, loc='upper right')

axes[0].set_ylabel('Altitude [km]', fontsize=13)
plt.tight_layout()
savepath = os.path.join(SAVE_DIR, 'one_step_rmse_vs_z.png')
fig.savefig(savepath, dpi=200, bbox_inches='tight', pad_inches=0.2)
print(f"Saved -> {savepath}")
plt.show()